# LiDAR Point Cloud Visualization with PyBoreas and Plotly

This notebook demonstrates how to load a segment from pyboreas and visualize the LiDAR point cloud using plotly.

In [1]:
# Import necessary libraries
import plotly.graph_objects as go
import numpy as np
from pyboreas import BoreasDataset

root = "/data/boreas"
bd = BoreasDataset(root)
seq = bd.sequences[0]

In [ ]:
START_INDEX = 5432
indices = [START_INDEX + i for i in range(5)]

sample_points = []
for idx in indices:
    lidar_frame = seq.get_lidar(idx)
    sample_points.append(lidar_frame.points[:, :3])
    lidar_frame.unload_data()  # Memory reqs will keep increasing without this

In [ ]:
MAX_DIST = 40.0

filtered_points = []
for points in sample_points:
    norms = np.linalg.norm(points, axis=1)  # Compute L2 norm for each point
    filtered_points.append(points[norms < MAX_DIST])

In [4]:
def DrawLidarCloud(input_points: np.ndarray, n_points: int = 10000):
    points = input_points[
        np.random.choice(input_points.shape[0], n_points, replace=False)
    ]

    # Create a 3D scatter plot for the LiDAR point cloud
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:, 0],
                y=points[:, 1],
                z=points[:, 2],
                mode="markers",
                marker=dict(
                    size=2,
                    color=points[:, 2],  # Color points by height
                    colorscale="Viridis",
                    opacity=0.7,
                ),
                name="LiDAR Points",
            )
        ]
    )

    # Update layout with custom styling
    fig.update_layout(
        title=dict(
            text="LiDAR Point Cloud Visualization", font=dict(size=20, family="Arial")
        ),
        scene=dict(
            xaxis_title="X (m)",
            yaxis_title="Y (m)",
            zaxis_title="Z (m)",
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectmode="data",
        ),
        height=900,
        width=1200,
    )
    return fig

## Visualize one frame

In [5]:
fig = DrawLidarCloud(filtered_points[0])
# Show the plot
fig.show()